In [1]:
import random
import zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

ARCHIVE_PATH = Path('/content/drive/MyDrive/Datasets/HERIDAL-YOLO.v2i.yolov8.zip')
EXTRACT_DIR = Path('data/heridal_yolo')

Mounted at /content/drive


In [2]:
def prepare_dataset(archive_path: Path, extract_dir: Path) -> Path:
    if not archive_path.exists():
        raise FileNotFoundError(f'Не найден архив: {archive_path.resolve()}')
    if not zipfile.is_zipfile(archive_path):
        raise RuntimeError('Архив пока невалидный как zip. Дождись окончания загрузки файла и запусти снова.')

    if not extract_dir.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path, 'r') as zf:
            zf.extractall(extract_dir)

    yaml_candidates = list(extract_dir.rglob('data.yaml'))
    if len(yaml_candidates) == 0:
        raise RuntimeError('В распакованном датасете не найден data.yaml')

    return yaml_candidates[0].parent
print(prepare_dataset(archive_path=ARCHIVE_PATH, extract_dir=EXTRACT_DIR))

data/heridal_yolo


In [3]:
def read_data_yaml(yaml_path: Path):
    names = {}
    for line in yaml_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        if line.startswith('- '):
            idx = len(names)
            names[idx] = line[2:].strip().strip('"').strip("'")
        elif ':' in line and line.split(':', 1)[0].strip().isdigit():
            idx_str, val = line.split(':', 1)
            names[int(idx_str.strip())] = val.strip().strip('"').strip("'")
    if len(names) == 0:
        names = {0: 'person'}
    return names

In [4]:
def parse_yolo_label(label_path: Path):
    boxes = []
    if not label_path.exists():
        return boxes

    for line in label_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        class_id = int(float(parts[0]))
        x_center = float(parts[1])
        y_center = float(parts[2])
        width = float(parts[3])
        height = float(parts[4])
        boxes.append((class_id, x_center, y_center, width, height))
    return boxes

In [5]:
def yolo_to_xyxy(box, image_w, image_h):
  class_id, x_center, y_center, width, height = box
  x1 = int((x_center - width / 2) * image_w)
  y1 = int((y_center - height / 2) * image_h)
  x2 = int((x_center + width / 2) * image_w)
  y2 = int((y_center + height / 2) * image_h)

  x1 = max(0, min(x1, image_w))
  y1 = max(0, min(y1, image_h))
  x2 = max(0, min(x2, image_w))
  y2 = max(0, min(y2, image_h))
  return class_id, x1, y1, x2, y2

In [6]:
def collect_split_stats(dataset_root: Path, split: str) -> dict:
    """Собирает статистики по одному split (train / valid / test).

    Возвращает словарь:
        n_images       – число изображений
        n_labels       – число txt-файлов с разметкой
        n_missing      – изображений без .txt
        n_empty        – пустых label-файлов
        boxes_per_img  – количество боксов на каждое изображение
        avg_boxes      – среднее число боксов
        max_boxes      – максимальное число боксов
    """
    images_dir = dataset_root / split / 'images'
    labels_dir = dataset_root / split / 'labels'

    image_paths = sorted(images_dir.glob('*.*')) if images_dir.exists() else []
    label_paths = sorted(labels_dir.glob('*.txt')) if labels_dir.exists() else []

    n_images  = len(image_paths)
    n_labels  = len(label_paths)

    n_missing = 0
    n_empty   = 0
    boxes_per_img = []

    for img_path in image_paths:
        lbl_path = labels_dir / f'{img_path.stem}.txt'
        if not lbl_path.exists():
            n_missing += 1
            boxes_per_img.append(0)
        else:
            boxes = parse_yolo_label(lbl_path)
            if len(boxes) == 0:
                n_empty += 1
            boxes_per_img.append(len(boxes))

    avg_boxes = sum(boxes_per_img) / len(boxes_per_img) if boxes_per_img else 0.0
    max_boxes = max(boxes_per_img) if boxes_per_img else 0

    return {
        'n_images'      : n_images,
        'n_labels'      : n_labels,
        'n_missing'     : n_missing,
        'n_empty'       : n_empty,
        'boxes_per_img' : boxes_per_img,
        'avg_boxes'     : avg_boxes,
        'max_boxes'     : max_boxes,
    }

In [7]:
def print_dataset_overview(dataset_root: Path, class_names: dict):
    """Печатает сводную таблицу статистик по всем split-ам датасета."""
    splits = ['train', 'valid', 'test']

    print(f'Датасет: {dataset_root}')
    print(f'Классы : {class_names}')
    print()
    print(f"{'Split':<8} {'Изображений':>12} {'Разметок':>10} {'Нет разметки':>14} {'Пусто':>7} {'Avg bbox':>9} {'Max bbox':>9}")
    print('-' * 75)

    for split in splits:
        split_dir = dataset_root / split
        if not split_dir.exists():
            print(f'{split:<8}  (нет папки)')
            continue

        s = collect_split_stats(dataset_root, split)
        print(
            f"{split:<8} {s['n_images']:>12} {s['n_labels']:>10} "
            f"{s['n_missing']:>14} {s['n_empty']:>7} "
            f"{s['avg_boxes']:>9.2f} {s['max_boxes']:>9}"
        )
    print()


In [8]:
def show_examples(dataset_root: Path, class_names: dict, split='train', n=6, seed=42):
    images_dir = dataset_root / split / 'images'
    labels_dir = dataset_root / split / 'labels'

    image_paths = sorted(list(images_dir.glob('*.*')))
    if len(image_paths) == 0:
        raise RuntimeError(f'В {images_dir} не найдено изображений')

    random.seed(seed)
    chosen = random.sample(image_paths, min(n, len(image_paths)))

    cols = 3
    rows = (len(chosen) + cols - 1) // cols
    plt.figure(figsize=(15, 5 * rows))

    for i, img_path in enumerate(chosen, start=1):
        label_path = labels_dir / f'{img_path.stem}.txt'

        with Image.open(img_path) as image:
            image = image.convert('RGB')
            image_w, image_h = image.size
            boxes = parse_yolo_label(label_path)

            ax = plt.subplot(rows, cols, i)
            ax.imshow(image)
            ax.set_title(f'{split}: | боксов: {len(boxes)}')
            ax.axis('off')

            for box in boxes:
                class_id, x1, y1, x2, y2 = yolo_to_xyxy(box, image_w, image_h)
                rect = patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    linewidth=2,
                    edgecolor='lime',
                    facecolor='none',
                )
                ax.add_patch(rect)
                class_name = class_names.get(class_id, f'class_{class_id}')
                ax.text(
                    x1,
                    max(0, y1 - 3),
                    class_name,
                    color='black',
                    fontsize=9,
                    bbox=dict(facecolor='lime', alpha=0.7),
                )

    plt.tight_layout()
    plt.show()


In [9]:
def print_yolo_train_guide(dataset_root: Path):
    data_yaml = dataset_root / 'data.yaml'

    print('Ссылка на документацию по обучению YOLO:')
    print('https://docs.ultralytics.com/modes/train/')
    print('')
    print('Команды для Google Colab:')
    print('pip install -U ultralytics')
    print('А остальное найти в статье ;)')

In [10]:
def main():
    dataset_root = prepare_dataset(ARCHIVE_PATH, EXTRACT_DIR)
    class_names = read_data_yaml(dataset_root / 'data.yaml')

    print_dataset_overview(dataset_root, class_names)
    print_yolo_train_guide(dataset_root)

    show_examples(dataset_root, class_names, split='train', n=6, seed=42)
    show_examples(dataset_root, class_names, split='valid', n=6, seed=43)
    show_examples(dataset_root, class_names, split='test', n=6, seed=44)


if __name__ == '__main__':
    main()


Output hidden; open in https://colab.research.google.com to view.

In [11]:
!pip install -q -U ultralytics

In [12]:
from ultralytics import YOLO

dataset_root = prepare_dataset(ARCHIVE_PATH, EXTRACT_DIR)
DATA_YAML = str(dataset_root / 'data.yaml')

# Веса и логи сохраняем на Google Drive, чтобы не потерять их при отключении Colab
RUNS_DIR = Path('/content/drive/MyDrive/yolo_runs')
RUN_DIR = RUNS_DIR / 'heridal'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'
BEST_PT = RUN_DIR / 'weights' / 'best.pt'

if LAST_PT.exists():
    # Colab отключился посреди обучения: продолжаем с последней эпохи
    print(f'Найден {LAST_PT}, продолжаю обучение')
    try:
        YOLO(str(LAST_PT)).train(resume=True)
    except AssertionError as e:
        # Ultralytics так сообщает, что обучение уже завершено
        print(f'Продолжать нечего: {e}')
else:
    # Загружаем предобученную модель (transfer learning, не с нуля)
    model = YOLO('yolov8s.pt')
    model.train(
        data=DATA_YAML,
        imgsz=1280,        # высокое разрешение — люди на аэрофото мелкие
        batch=8,           # под T4 (16 GB VRAM) при imgsz=1280
        epochs=100,
        patience=20,       # early stopping, если 20 эпох без улучшения mAP
        device=0,          # GPU
        project=str(RUNS_DIR),
        name='heridal',
        exist_ok=True,
    )

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Найден /content/drive/MyDrive/yolo_runs/heridal/weights/last.pt, продолжаю обучение
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/heridal_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, eras

## Оценка на тестовой выборке

Лучшую эпоху выбирали по valid, поэтому честная итоговая оценка — только на test, который модель не видела.

In [13]:
best = YOLO(str(BEST_PT))
m = best.val(data=DATA_YAML, split='test', imgsz=1280, batch=8, device=0)
print(f'TEST: mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}  P={m.box.mp:.3f}  R={m.box.mr:.3f}')

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 70.2±28.9 MB/s, size: 237.3 KB)
val: Scanning /content/data/heridal_yolo/test/labels... 154 images, 45 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 154/154 573.1it/s 0.3s
val: New cache created: /content/data/heridal_yolo/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 1.9it/s 10.8s
                   all        154        286      0.848      0.782      0.849      0.474
Speed: 8.3ms preprocess, 51.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/detect/val
TEST: mAP50=0.849  mAP50-95=0.474  P=0.848  R=0.782


In [14]:
results = best.predict(
    source=str(dataset_root / 'test' / 'images'),
    imgsz=1280,
    conf=0.25,
    save=True,
    project=str(RUNS_DIR),
    name='heridal_predict',
    exist_ok=True,
)


image 1/154 /content/data/heridal_yolo/test/images/train_BRA_1002_JPG.rf.07a8199628b09a6d47d9b43be19712bb.jpg: 1280x1280 4 humans, 81.2ms
image 2/154 /content/data/heridal_yolo/test/images/train_BRA_1003_JPG.rf.14fc93d1421f25f61b7387a81dfb7487.jpg: 1280x1280 (no detections), 53.6ms
image 3/154 /content/data/heridal_yolo/test/images/train_BRA_1006_JPG.rf.c14c76eae0e8bf50b1124722d9278081.jpg: 1280x1280 (no detections), 53.6ms
image 4/154 /content/data/heridal_yolo/test/images/train_BRA_1016_JPG.rf.6e5d4f9650fb964e4e05ba238c86ba92.jpg: 1280x1280 1 human, 49.0ms
image 5/154 /content/data/heridal_yolo/test/images/train_BRA_1017_JPG.rf.03be1b8c8ef103b4abc78f20985816db.jpg: 1280x1280 1 human, 48.8ms
image 6/154 /content/data/heridal_yolo/test/images/train_BRA_1039_JPG.rf.11b96a843fb849cbb5a27e02054b2d14.jpg: 1280x1280 (no detections), 49.0ms
image 7/154 /content/data/heridal_yolo/test/images/train_BRA_1071_JPG.rf.0d112a6a3817c8fad53e824075da9061.jpg: 1280x1280 11 humans, 48.5ms
image 8/154 /